# Clase 5 — Waffle charts, Word Clouds y Seaborn

En esta clase vemos tres herramientas para dar impacto visual y resumir estadística rápidamente:

1. **Waffle charts** con la librería `pywaffle` (alternativa a un pie chart).
2. **Word clouds** con la librería `wordcloud` (qué palabras dominan un texto).
3. **Seaborn**: `countplot`, `barplot` y `regplot`, comparados con el scatter + `polyfit` manual visto en clases anteriores.

> Nota: las URLs de datos (`Canada.csv`, `alice_novel.txt`, `alice_mask.png`) son las mismas que usa el notebook oficial del curso (IBM Skills Network), así que este notebook corre sin necesidad de archivos locales.

In [ ]:
# instalar las dos dependencias extra que usamos en esta clase
!pip install pywaffle wordcloud

In [ ]:
# Import de librerías
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from pywaffle import Waffle
from wordcloud import WordCloud, STOPWORDS
from PIL import Image
import urllib

mpl.style.use('ggplot')
%matplotlib inline

print('Librerías importadas correctamente')

## 1. Cargar el dataset de inmigración a Canadá

Usamos el mismo dataset limpio (`Canada.csv`) del notebook oficial del curso, alojado en el bucket público de IBM Skills Network.

In [ ]:
df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')

# poner 'Country' como índice, como en clases anteriores
df_can.set_index('Country', inplace=True)

print('Forma del dataset:', df_can.shape)
df_can.head()

## 2. Waffle charts

### La idea

Un waffle chart es como un pie chart, pero en forma de cuadrícula: se divide un rectángulo en celdas (por ejemplo 20x30 = 600 celdas) y se colorea cada celda según la proporción que le corresponde a cada categoría.

### El método manual (solo para entender qué pasa por dentro)

Construirlo a mano implica: calcular proporciones, definir la cuadrícula, rellenar una matriz de NumPy, dibujarla con `plt.matshow()` y fabricar una leyenda a mano. Es instructivo, pero nadie lo repite cada vez — para eso existe `pywaffle`.

### Con `pywaffle` (la forma que usarás en la práctica)

Recreamos el caso de Dinamarca, Noruega y Suecia.

In [ ]:
# subconjunto de países escandinavos
df_dsn = df_can.loc[['Denmark', 'Norway', 'Sweden'], :]
df_dsn[['Total']]

In [ ]:
# waffle chart con pywaffle: una sola llamada en vez de ~40 líneas manuales
fig = plt.figure(
    FigureClass=Waffle,
    rows=20,            # número de filas de la cuadrícula
    columns=30,          # número de columnas de la cuadrícula
    values=df_dsn['Total'],   # los datos: cuánto ocupa cada categoría
    cmap_name='tab20',        # esquema de color
    legend={
        'labels': [f"{k} ({v})" for k, v in zip(df_dsn.index.values, df_dsn.Total)],
        'loc': 'lower left',
        'bbox_to_anchor': (0, -0.1),
        'ncol': 3
    }
)

plt.show()

Fíjate en la línea de la leyenda: la *list comprehension* construye automáticamente etiquetas como `"Sweden (5866)"` combinando el índice del país con su total. No hace falta escribirlas a mano.

## 3. Word clouds

### La idea

Una nube de palabras muestra qué palabras dominan un texto: cuanto más se repite una palabra, más grande y en negrita se dibuja. Usamos la librería `wordcloud` (de Andreas Mueller) sobre el texto completo de *Alice's Adventures in Wonderland* (Lewis Carroll), descargado desde la misma URL pública que usa el notebook oficial del curso.

In [ ]:
# descargar el texto completo de la novela
alice_novel = urllib.request.urlopen('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/alice_novel.txt').read().decode("utf-8")

# stopwords: palabras vacías ("the", "and", etc.) que no aportan información
stopwords = set(STOPWORDS)

# instanciar el objeto WordCloud y generarlo a partir del texto
alice_wc = WordCloud(background_color='white', stopwords=stopwords)
alice_wc.generate(alice_novel)

# WordCloud no se dibuja solo: necesitamos matplotlib para mostrarlo
plt.figure(figsize=(14, 18))
plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')   # una nube de palabras no necesita ejes
plt.show()

Si ejecutas esto sin filtrar nada, palabras como **"said"** dominan la imagen sin aportar información real. La solución es agregarla manualmente a `stopwords` según lo que veas y regenerar:

In [ ]:
stopwords.add('said')  # 'said' aparece muchísimo pero no dice nada interesante
alice_wc.generate(alice_novel)  # regenerar con el nuevo filtro

plt.figure(figsize=(14, 18))
plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')
plt.show()

### Nube de palabras con máscara (mask)

`WordCloud` acepta un parámetro `mask` con una imagen para que las palabras se acomoden dentro de esa forma en lugar de un rectángulo. Usamos la silueta de Alice y el conejo, también descargada desde la URL pública del notebook oficial.

In [ ]:
# descargar y convertir la máscara en un array de NumPy
alice_mask = np.array(Image.open(urllib.request.urlopen('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%204/images/alice_mask.png')))

alice_wc = WordCloud(background_color='white', max_words=2000,
                      mask=alice_mask, stopwords=stopwords)
alice_wc.generate(alice_novel)

fig = plt.figure(figsize=(14, 18))
plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')
plt.show()

### Word cloud a partir de datos numéricos

También podemos generar una nube de palabras a partir de datos que no son texto narrativo: repetimos el nombre de cada país de `df_can` un número de veces proporcional a su peso en el total de inmigración, y pasamos ese texto generado a `WordCloud().generate()`.

In [ ]:
total_immigration = df_can['Total'].sum()

max_words = 90
word_string = ''
for country in df_can.index.values:
    # solo países con nombre de una sola palabra (simplifica el ejemplo)
    if country.count(" ") == 0:
        repeat_num_times = int(df_can.loc[country, 'Total'] / total_immigration * max_words)
        word_string = word_string + ((country + ' ') * repeat_num_times)

wordcloud_countries = WordCloud(background_color='white').generate(word_string)

plt.figure(figsize=(14, 18))
plt.imshow(wordcloud_countries, interpolation='bilinear')
plt.axis('off')
plt.show()

## 4. Seaborn: estadística visual

Seaborn está construido sobre Matplotlib y ya conoce las convenciones de la estadística visual: conteo por categoría, agregación con intervalos de confianza, y ajuste + dibujo de una regresión, todo con menos código que el necesario en Matplotlib puro.

### `countplot`: cuenta filas por categoría

In [ ]:
df_can.reset_index(inplace=True)  # para tener 'Country' como columna otra vez, si hiciera falta

# ¿cuántos países hay registrados por continente?
plt.figure(figsize=(15, 10))
sns.countplot(x='Continent', data=df_can)
plt.show()

Si las etiquetas del eje x quedan muy largas, podemos limpiarlas antes con `replace`:

In [ ]:
df_can1 = df_can.replace('Latin America and the Caribbean', 'L-America')
df_can1 = df_can1.replace('Northern America', 'N-America')

plt.figure(figsize=(15, 10))
sns.countplot(x='Continent', data=df_can1)
plt.show()

### `barplot`: agrega un valor numérico por categoría

Mientras `countplot` cuenta *filas*, `barplot` agrega un *valor numérico* (por defecto la media) agrupando por una variable categórica, y añade automáticamente una barra de error con el intervalo de confianza.

In [ ]:
plt.figure(figsize=(15, 10))
sns.barplot(x='Continent', y='Total', data=df_can1)
plt.show()

# podemos verificar el cálculo nosotros mismos
df_can1.groupby('Continent')['Total'].mean()

### `regplot`: scatter + regresión en una sola línea

Esta es la comparación que más vale la pena: en clases anteriores, para dibujar un scatter con línea de tendencia usábamos `np.polyfit` + `np.poly1d` + `plt.scatter` + `plt.plot` (~20 líneas). Con Seaborn es una sola llamada.

In [ ]:
years = list(map(str, range(1980, 2014)))

# total de inmigración por año (todas las filas sumadas)
df_tot = pd.DataFrame(df_can[years].sum(axis=0))
df_tot.index = map(float, df_tot.index)
df_tot.reset_index(inplace=True)
df_tot.columns = ['year', 'total']

df_tot.head()

In [ ]:
# scatter + línea de regresión ajustada + banda de confianza, en una sola línea
sns.regplot(x='year', y='total', data=df_tot)
plt.show()

In [ ]:
# personalización: color, marcador, tamaño de figura, título y ejes
plt.figure(figsize=(15, 10))
sns.set(font_scale=1.5)
sns.set_style('whitegrid')

ax = sns.regplot(x='year', y='total', data=df_tot, color='green',
                  marker='+', scatter_kws={'s': 200})

ax.set(xlabel='Year', ylabel='Total Immigration')
ax.set_title('Total Immigration to Canada from 1980 - 2013')
plt.show()

## Puntos clave

- Un **waffle chart** es una alternativa visual a un pie chart; con `pywaffle` se crea en una sola llamada usando `Waffle` como `FigureClass` de matplotlib.
- Una **nube de palabras** (`WordCloud` + `.generate()` + `plt.imshow()`) muestra qué palabras dominan un texto; `STOPWORDS` filtra palabras irrelevantes y `mask` permite darle una forma personalizada.
- **Seaborn** (`countplot`, `barplot`, `regplot`) reemplaza mucho código manual de Matplotlib con funciones de alto nivel pensadas para relaciones estadísticas.
- `sns.set()` y `sns.set_style()` controlan el aspecto general de todos los gráficos de la sesión sin tocarlos uno por uno.